# 02 QC: Compare coverage between sequencing runs

Comparing sequencing coverage across Run1, Run2, Run3, and Run4 using the merged sequencing metrics file.

It creates overlaid density plots for:
- `MEAN_TARGET_COVERAGE`
- `MEDIAN_TARGET_COVERAGE`

Plots are generated for all samples and after optional mean coverage cutoffs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

metrics_csv = Path(r"C:\Users\katya\Box\KD_SUIP_noncoding_project\sample_info\sequencing_metrics_merged_with_grouplabels.csv")
output_base = Path(r"C:\Users\katya\Box\KD_SUIP_noncoding_project\sequencing_and_variantcalling_overview\comparison")
output_base.mkdir(parents=True, exist_ok=True)

output_prefix = "02_"

metrics = pd.read_csv(metrics_csv)
metrics.head()

## Settings
Coverage plots are generated for all samples and after optional mean coverage cutoffs.

`None` means no coverage filter is applied.

In [ ]:
runs_to_process = ["Run1", "Run2", "Run3", "Run4"]
coverage_cutoffs = [None, 1, 20, 30]

coverage_columns = [
    "MEAN_TARGET_COVERAGE",
    "MEDIAN_TARGET_COVERAGE",
]

## Run counts

Check how many samples are assigned to each run before plotting.

In [ ]:
metrics["Group"].value_counts()

## Coverage density plotting function

This function overlays coverage density curves across runs.

The optional cutoff filters samples using `MEAN_TARGET_COVERAGE`.

In [ ]:
def plot_run_overlay_density(metrics_df, value_col, mean_cov_filter=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    for run in runs_to_process:
        df_run = metrics_df[metrics_df["Group"].astype(str).str.strip() == run]

        if mean_cov_filter is not None:
            df_run = df_run[df_run["MEAN_TARGET_COVERAGE"] >= mean_cov_filter]

        values = df_run[value_col].dropna().astype(float)

        if len(values) < 2:
            print(f"Skipping {run} for {value_col}: not enough data")
            continue

        x_grid = np.linspace(values.min(), values.max(), 500)
        y = gaussian_kde(values)(x_grid)

        ax.plot(x_grid, y, label=f"{run} (n={len(values)})")

    cutoff_label = "all samples" if mean_cov_filter is None else f"MEAN_TARGET_COVERAGE >= {mean_cov_filter}"

    ax.set_xlabel(value_col)
    ax.set_ylabel("Density")
    ax.set_title(f"{value_col} overlay by run ({cutoff_label})")
    ax.legend()
    fig.tight_layout()

    file_label = "all_samples" if mean_cov_filter is None else f"mean_filter_{mean_cov_filter}"
    output_path = output_base / f"{output_prefix}all_runs_{value_col.lower()}_overlay_{file_label}.png"

    fig.savefig(output_path, dpi=300)
    plt.show()
    plt.close(fig)

## Generate coverage comparison plots

Mean and median target coverage plots for all samples and for each mean coverage cutoff.

In [ ]:
for cutoff in coverage_cutoffs:
    for col in coverage_columns:
        plot_run_overlay_density(
            metrics_df=metrics,
            value_col=col,
            mean_cov_filter=cutoff,
        )